In [1]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 \
    transformers accelerate bitsandbytes sacrebleu rapidfuzz \
    bert-score==0.3.13

from google.colab import drive
drive.mount("/content/drive")

import os, glob, json, re, pickle, unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
import torch

OUT = Path("/content/drive/MyDrive/Bangla_Agri_RAG")
OUT.mkdir(parents=True, exist_ok=True)

def find_file(pattern):
    files = glob.glob("/content/" + pattern)
    if not files:
        raise FileNotFoundError(pattern)
    return max(files, key=os.path.getmtime)

KB_FILE = find_file("Bangla_Agriculture_Knowledge_Base_500*.json")
TRAIN_FILE = find_file("Bangla_Agriculture_QA_Train_800*.json")
TEST_FILE = find_file("Bangla_Agriculture_QA_Test_200*.json")

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def chunk_text(text, size=180, overlap=30):
    words = str(text).split()

    if len(words) <= size:
        return [str(text).strip()]

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + size, len(words))
        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break

        start = end - overlap

    return chunks

kb_rows = load_json(KB_FILE)

chunks = []

for row in kb_rows:

    text = str(row.get("content", "")).strip()
    if not text:
        continue

    doc_id = str(row.get("id", f"kb_{len(chunks)}"))

    for part, text_chunk in enumerate(chunk_text(text)):

        title = str(row.get("title", "")).strip()
        source_title = str(row.get("source_title", "")).strip()

        index_text = (
            f"শিরোনাম: {title}\n"
            f"বিষয়: {source_title}\n"
            f"তথ্য: {text_chunk}"
        )

        chunks.append({
            "chunk_id": len(chunks),
            "doc_id": doc_id,
            "title": title,
            "source_site": str(row.get("source_site", "")),
            "source_title": source_title,
            "source_url": str(row.get("source_url", "")),
            "text": text_chunk,
            "index_text": index_text
        })

def load_qa(path):
    data = load_json(path)
    rows = data["qa_pairs"] if isinstance(data, dict) and "qa_pairs" in data else data
    out = []
    for row in rows:
        out.append({
            "id": str(row.get("id", "")),
            "question": str(row.get("question", "")).strip(),
            "gold": str(row.get("reference_answer", "")).strip()
        })
    return out

train = load_qa(TRAIN_FILE)
tests = load_qa(TEST_FILE)

train_df = pd.DataFrame(train)
test_df = pd.DataFrame(tests)

ROUTER_CALIB_FRACTION = 0.20
ROUTER_SPLIT_SEED = 42

calib_n = max(1, int(round(len(train_df) * ROUTER_CALIB_FRACTION)))
router_calib_df = train_df.sample(
    n=calib_n,
    random_state=ROUTER_SPLIT_SEED
).sort_index().reset_index(drop=True)

router_dev_df = train_df.drop(
    index=train_df.sample(
        n=calib_n,
        random_state=ROUTER_SPLIT_SEED
    ).index
).reset_index(drop=True)

with open(OUT / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

test_df.to_csv(
    OUT / "test_questions.csv",
    index=False,
    encoding="utf-8-sig"
)

train_df.to_csv(
    OUT / "train_questions.csv",
    index=False,
    encoding="utf-8-sig"
)

router_calib_df.to_csv(
    OUT / "router_calibration_questions.csv",
    index=False,
    encoding="utf-8-sig"
)

router_dev_df.to_csv(
    OUT / "router_development_questions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("KB file:", os.path.basename(KB_FILE))
print("Train file:", os.path.basename(TRAIN_FILE))
print("Test file:", os.path.basename(TEST_FILE))

print("\nTotal chunks:", len(chunks))
print("Total train questions:", len(train_df))
print("Router development questions:", len(router_dev_df))
print("Router calibration questions:", len(router_calib_df))
print("Total test questions:", len(test_df))
print("Saved to:", OUT)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.0 MB/s eta 0:00:00
Mounted at /content/drive
KB file: Bangla_Agriculture_Knowledge_Base_500.json
Train file: Bangla_Agriculture_QA_Train_800.json
Test file: Bangla_Agriculture_QA_Test_200.json

Total chunks: 500
Total train questions: 800
Router development questions: 640
Router calibration questions: 160
Total test questions: 200
Saved to: /content/drive/MyDrive/Bangla_Agri_RAG


In [2]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
import gc

with open(OUT / "chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [x["index_text"] for x in chunks]

device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer(
    "BAAI/bge-m3",
    device=device
)

embeddings = embedder.encode(
    texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(
    index,
    str(OUT / "bge_m3.faiss")
)

np.save(
    OUT / "bge_m3_embeddings.npy",
    embeddings
)

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)

def tokenize(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(BN_TO_EN).lower()

    return re.findall(
        r"[\u0980-\u09FF]+|[a-z]+|\d+(?:\.\d+)?",
        text
    )

tokenized_corpus = [tokenize(x) for x in texts]

bm25 = BM25Okapi(tokenized_corpus)

with open(OUT / "bm25.pkl", "wb") as f:
    pickle.dump({
        "bm25": bm25,
        "tokens": tokenized_corpus
    }, f)

print("FAISS vectors:", index.ntotal)
print("BM25 documents:", bm25.corpus_size)

del embedder, embeddings
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Indices saved successfully.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

FAISS vectors: 500
BM25 documents: 500
Indices saved successfully.


In [3]:
import json, pickle, gc, re, unicodedata
import numpy as np
import pandas as pd
import torch
import faiss

from pathlib import Path
from collections import defaultdict, Counter
from tqdm.auto import tqdm
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

OUT = Path("/content/drive/MyDrive/Bangla_Agri_RAG")


with open(OUT / "chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

tests = pd.read_csv(OUT / "test_questions.csv").fillna("")

train_full = pd.read_csv(OUT / "train_questions.csv").fillna("") \
    if (OUT / "train_questions.csv").exists() else None

router_calib_path = OUT / "router_calibration_questions.csv"

if router_calib_path.exists():
    router_calib = pd.read_csv(router_calib_path).fillna("")
elif train_full is not None and len(train_full) > 0:
    calib_n = max(1, int(round(len(train_full) * 0.20)))
    router_calib = train_full.sample(
        n=calib_n,
        random_state=42
    ).reset_index(drop=True)
else:
    raise FileNotFoundError(
        "Router calibration data not found. Re-run Cell 1 so that "
        "train_questions.csv and router_calibration_questions.csv are created."
    )

faiss_index = faiss.read_index(str(OUT / "bge_m3.faiss"))

with open(OUT / "bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)["bm25"]

BN_TO_EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def tokenize(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(BN_TO_EN).lower()
    return re.findall(r"[\u0980-\u09FF]+|[a-z]+|\d+(?:\.\d+)?", text)

def normalize(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.translate(BN_TO_EN).lower()
    text = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def token_f1_calib(pred, gold):
    p = normalize(pred).split()
    g = normalize(gold).split()

    if not p or not g:
        return 0.0

    overlap = sum((Counter(p) & Counter(g)).values())
    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)
    return 2 * precision * recall / (precision + recall)

device = "cuda" if torch.cuda.is_available() else "cpu"

embedder = SentenceTransformer("BAAI/bge-m3", device=device)

reranker_name = "BAAI/bge-reranker-v2-m3"
rerank_tokenizer = AutoTokenizer.from_pretrained(reranker_name)
reranker = AutoModelForSequenceClassification.from_pretrained(
    reranker_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
reranker.eval()


def hybrid_retrieve(question, candidate_k=25):
    qvec = embedder.encode(
        [question], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")

    dense_scores, dense_ids = faiss_index.search(qvec, min(100, faiss_index.ntotal))
    dense_rank = [(int(idx), float(score)) for score, idx in zip(dense_scores[0], dense_ids[0])]

    bm25_scores = bm25.get_scores(tokenize(question))
    sparse_ids = np.argsort(bm25_scores)[::-1][:100]
    sparse_rank = [(int(idx), float(bm25_scores[idx])) for idx in sparse_ids]

    rrf = defaultdict(float)
    dense_map, bm25_map = {}, {}
    for rank, (idx, score) in enumerate(dense_rank, 1):
        rrf[idx] += 1 / (60 + rank)
        dense_map[idx] = score
    for rank, (idx, score) in enumerate(sparse_rank, 1):
        rrf[idx] += 1 / (60 + rank)
        bm25_map[idx] = score

    best_ids = sorted(rrf, key=rrf.get, reverse=True)[:candidate_k]
    candidates = []
    for idx in best_ids:
        item = chunks[idx].copy()
        item["dense_score"] = dense_map.get(idx, 0)
        item["bm25_score"] = bm25_map.get(idx, 0)
        item["rrf_score"] = rrf[idx]
        candidates.append(item)
    return candidates


def score_pairs(question, texts, batch_size=8):
    scores = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = rerank_tokenizer(
            [question] * len(batch), batch,
            padding=True, truncation=True, max_length=512,
            return_tensors="pt"
        ).to(device)
        with torch.inference_mode():
            logits = reranker(**encoded).logits.squeeze(-1)
        scores.extend(logits.float().cpu().tolist())
    return scores

def rerank_chunks(question, candidates, top_k=3):
    scores = score_pairs(question, [x["index_text"] for x in candidates])
    for item, score in zip(candidates, scores):
        item["reranker_score"] = float(score)
    return sorted(candidates, key=lambda x: x["reranker_score"], reverse=True)[:top_k]


def split_sentences(text):
    parts = re.split(r"(?<=।)\s*", str(text).strip())
    return [p.strip() for p in parts if p.strip()]

def best_extractive_answer(question, docs):
    candidates = []
    for doc in docs:
        for sent in split_sentences(doc["text"]):
            candidates.append(sent)
    if not candidates:
        return None, float("-inf"), 0.0

    scores = score_pairs(question, candidates)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    best_sent, best_score = ranked[0]
    second_score = ranked[1][1] if len(ranked) > 1 else best_score
    return best_sent, best_score, best_score - second_score


calib_retrieval_cache = []
calib_extractive_cache = []

for _, row in tqdm(
    router_calib.iterrows(),
    total=len(router_calib),
    desc="Router calibration retrieval + extraction"
):
    cands = hybrid_retrieve(row["question"])
    docs = rerank_chunks(row["question"], cands, top_k=3)
    sent, score, margin = best_extractive_answer(row["question"], docs)
    calib_retrieval_cache.append(docs)
    calib_extractive_cache.append({
        "sentence": sent,
        "score": score,
        "margin": margin
    })

retrieval_cache = []
extractive_cache = []

for _, row in tqdm(tests.iterrows(), total=len(tests), desc="Hybrid retrieval + extraction"):
    cands = hybrid_retrieve(row["question"])
    docs = rerank_chunks(row["question"], cands, top_k=3)
    sent, score, margin = best_extractive_answer(row["question"], docs)
    retrieval_cache.append(docs)
    extractive_cache.append({"sentence": sent, "score": score, "margin": margin})

retrieval_rows = []
for row_idx, row in tests.iterrows():
    docs = retrieval_cache[row_idx]
    x = {"id": row["id"], "question": row["question"]}
    for i, doc in enumerate(docs, 1):
        x[f"doc_{i}"] = doc["doc_id"]
        x[f"title_{i}"] = doc["title"]
        x[f"context_{i}"] = doc["text"]
        x[f"source_{i}"] = doc["source_url"]
        x[f"reranker_score_{i}"] = doc["reranker_score"]
    x["extractive_sentence"] = extractive_cache[row_idx]["sentence"]
    x["extractive_score"] = extractive_cache[row_idx]["score"]
    retrieval_rows.append(x)

pd.DataFrame(retrieval_rows).to_csv(OUT / "retrievals.csv", index=False, encoding="utf-8-sig")

with open(OUT / "retrievals.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_cache, f, ensure_ascii=False, indent=2)

print("Retrievals + extractive candidates saved.")

del embedder, reranker, rerank_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer_qwen = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map="auto")
model.eval()
model_device = model.get_input_embeddings().weight.device

MAX_NEW_TOKENS = 160

FEWSHOT = [
    {
        "context": "[Context 1]\nশিরোনাম: আলু: সংরক্ষণ\nতথ্য: অস্থায়ী শেড থেকে বাছাই শেষে বিক্রির জন্য বিক্রয় করা অথবা সংরক্ষণের জন্য কোল্ড স্টোরে রাখা যাবে।",
        "question": "আলু কীভাবে সংরক্ষণ করতে হবে?",
        "answer": "অস্থায়ী শেড থেকে বাছাই শেষে বিক্রির জন্য বিক্রয় করা অথবা সংরক্ষণের জন্য কোল্ড স্টোরে রাখা যাবে।"
    },
    {
        "context": "[Context 1]\nশিরোনাম: ভুট্টা: জাত\nতথ্য: বারি হাইব্রিড ভুট্টা-১৪ জাতটি ২০১৭ সালে অবমুক্ত করা হয়। রবি মৌসুমে এর জীবনকাল প্রায় ১৪০ দিন।",
        "question": "বারি হাইব্রিড ভুট্টা-১৪ জাতটি কবে অবমুক্ত করা হয়?",
        "answer": "বারি হাইব্রিড ভুট্টা-১৪ জাতটি ২০১৭ সালে অবমুক্ত করা হয়।"
    }
]

SYSTEM_PROMPT = (
    "তুমি বাংলাদেশের কৃষি বিষয়ক একজন সহায়ক সহকারী। "
    "শুধু প্রদত্ত Context থেকে হুবহু (verbatim) একটি বাক্য বা তার অংশ কপি করে উত্তর দাও — "
    "নিজের ভাষায় ব্যাখ্যা বা প্যারাফ্রেজ কোরো না। "
    "উত্তর সর্বোচ্চ এক বা দুই বাক্যের মধ্যে সীমাবদ্ধ রাখো, কোনো ভূমিকা বা উপসংহার দিও না। "
    "Context-এ উত্তর একেবারেই না থাকলে শুধু বলো: 'প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।'"
)

def generate_answer_llm(question, docs):
    context = "\n\n".join([
        f"[Context {i}]\nশিরোনাম: {doc['title']}\nতথ্য: {doc['text']}"
        for i, doc in enumerate(docs, 1)
    ])

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in FEWSHOT:
        messages.append({"role": "user", "content": f"Context:\n{ex['context']}\n\nপ্রশ্ন: {ex['question']}\n\nউত্তর:"})
        messages.append({"role": "assistant", "content": ex["answer"]})
    messages.append({"role": "user", "content": f"Context:\n{context}\n\nপ্রশ্ন: {question}\n\nউত্তর:"})

    prompt = tokenizer_qwen.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer_qwen(prompt, return_tensors="pt", truncation=True, max_length=7000).to(model_device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer_qwen.eos_token_id,
            eos_token_id=tokenizer_qwen.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer_qwen.decode(generated, skip_special_tokens=True).strip()
    truncated = (len(generated) >= MAX_NEW_TOKENS and generated[-1].item() != tokenizer_qwen.eos_token_id)
    return answer, truncated


calib_rows = []

for i, row in tqdm(
    router_calib.iterrows(),
    total=len(router_calib),
    desc="Generating Qwen calibration answers"
):
    ex = calib_extractive_cache[i]
    gen_answer, gen_truncated = generate_answer_llm(
        row["question"],
        calib_retrieval_cache[i]
    )

    extractive_answer = ex["sentence"] or ""

    calib_rows.append({
        "id": row["id"],
        "question": row["question"],
        "gold": row["gold"],
        "extractive": extractive_answer,
        "generative": gen_answer,
        "score": float(ex["score"]),
        "margin": float(ex["margin"]),
        "generative_truncated": bool(gen_truncated),
        "extractive_token_f1": token_f1_calib(extractive_answer, row["gold"]),
        "generative_token_f1": token_f1_calib(gen_answer, row["gold"]),
        "extractive_fuzzy": fuzz.token_set_ratio(
            normalize(extractive_answer), normalize(row["gold"])
        ) / 100.0,
        "generative_fuzzy": fuzz.token_set_ratio(
            normalize(gen_answer), normalize(row["gold"])
        ) / 100.0
    })

calib_df = pd.DataFrame(calib_rows)

valid_scores = sorted(
    calib_df.loc[
        calib_df["extractive"].astype(str).str.len().gt(0)
        & np.isfinite(calib_df["score"]),
        "score"
    ].unique().tolist()
)

if not valid_scores:
    raise RuntimeError(
        "No valid extractive scores were produced on the router calibration split."
    )

thresholds = valid_scores + [float(np.nextafter(max(valid_scores), np.inf))]

search_rows = []

for t in thresholds:
    use_extractive = (
        calib_df["extractive"].astype(str).str.len().gt(0)
        & np.isfinite(calib_df["score"])
        & (calib_df["score"] >= t)
    )

    hybrid_token_f1 = np.where(
        use_extractive,
        calib_df["extractive_token_f1"],
        calib_df["generative_token_f1"]
    )

    hybrid_fuzzy = np.where(
        use_extractive,
        calib_df["extractive_fuzzy"],
        calib_df["generative_fuzzy"]
    )

    search_rows.append({
        "threshold": float(t),
        "hybrid_token_f1": float(np.mean(hybrid_token_f1)),
        "hybrid_fuzzy": float(np.mean(hybrid_fuzzy)),
        "extractive_count": int(use_extractive.sum()),
        "generative_count": int((~use_extractive).sum()),
        "extractive_rate": float(use_extractive.mean())
    })

threshold_df = pd.DataFrame(search_rows)

best_row = threshold_df.sort_values(
    ["hybrid_token_f1", "hybrid_fuzzy", "extractive_rate"],
    ascending=[False, False, False]
).iloc[0]

EXTRACTIVE_THRESHOLD = float(best_row["threshold"])

calib_df["route"] = np.where(
    calib_df["extractive"].astype(str).str.len().gt(0)
    & np.isfinite(calib_df["score"])
    & (calib_df["score"] >= EXTRACTIVE_THRESHOLD),
    "extractive",
    "generative"
)

calib_df["hybrid_prediction"] = np.where(
    calib_df["route"].eq("extractive"),
    calib_df["extractive"],
    calib_df["generative"]
)

calib_df["hybrid_token_f1"] = [
    token_f1_calib(p, g)
    for p, g in zip(calib_df["hybrid_prediction"], calib_df["gold"])
]

calib_df["hybrid_fuzzy"] = [
    fuzz.token_set_ratio(normalize(p), normalize(g)) / 100.0
    for p, g in zip(calib_df["hybrid_prediction"], calib_df["gold"])
]

calib_df.to_csv(
    OUT / "calibration_train_sample.csv",
    index=False,
    encoding="utf-8-sig"
)

threshold_df.to_csv(
    OUT / "router_threshold_search.csv",
    index=False,
    encoding="utf-8-sig"
)

all_extractive_f1 = calib_df["extractive_token_f1"].mean()
all_generative_f1 = calib_df["generative_token_f1"].mean()
hybrid_f1 = calib_df["hybrid_token_f1"].mean()
routed_pct = calib_df["route"].eq("extractive").mean() * 100

print(f"\nCalibrated EXTRACTIVE_THRESHOLD = {EXTRACTIVE_THRESHOLD:.6f}")
print(f"  -> calibration all-extractive Token F1: {all_extractive_f1:.4f}")
print(f"  -> calibration all-generative Token F1: {all_generative_f1:.4f}")
print(f"  -> calibration hybrid Token F1:         {hybrid_f1:.4f}")
print(f"  -> extractive routing rate:             {routed_pct:.1f}%")
print("  -> TEST labels were not used for threshold selection.")

predictions = []

for i, row in tqdm(tests.iterrows(), total=len(tests), desc="Answering"):
    ex = extractive_cache[i]

    use_extractive = (
        ex["sentence"] is not None
        and np.isfinite(ex["score"])
        and ex["score"] >= EXTRACTIVE_THRESHOLD
    )

    if use_extractive:
        answer, truncated, method = ex["sentence"], False, "extractive"
    else:
        answer, truncated = generate_answer_llm(row["question"], retrieval_cache[i])
        method = "generative"

    predictions.append({
        "id": row["id"],
        "question": row["question"],
        "gold": row["gold"],
        "prediction": answer,
        "truncated": truncated,
        "method": method,
        "extractive_score": ex["score"]
    })

    pd.DataFrame(predictions).to_csv(
        OUT / "predictions_partial.csv",
        index=False,
        encoding="utf-8-sig"
    )

pred_df = pd.DataFrame(predictions)
pred_df.to_csv(OUT / "predictions.csv", index=False, encoding="utf-8-sig")

print("\nCompleted:", len(pred_df))
print("Extractive:", (pred_df["method"] == "extractive").sum())
print("Generative (Qwen fallback):", (pred_df["method"] == "generative").sum())
print("Truncated:", int(pred_df["truncated"].sum()))
print("Saved:", OUT / "retrievals.csv")
print("Saved:", OUT / "predictions.csv")
print("Saved:", OUT / "calibration_train_sample.csv")
print("Saved:", OUT / "router_threshold_search.csv")

del model, tokenizer_qwen
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Generating Qwen calibration answers:   0%|          | 0/160 [00:00<?, ?it/s]


Calibrated EXTRACTIVE_THRESHOLD = 1.064453
  -> calibration all-extractive Token F1: 0.5915
  -> calibration all-generative Token F1: 0.6855
  -> calibration hybrid Token F1:         0.6935
  -> extractive routing rate:             36.9%
  -> TEST labels were not used for threshold selection.


Answering:   0%|          | 0/200 [00:00<?, ?it/s]


Completed: 200
Extractive: 87
Generative (Qwen fallback): 113
Truncated: 23
Saved: /content/drive/MyDrive/Bangla_Agri_RAG/retrievals.csv
Saved: /content/drive/MyDrive/Bangla_Agri_RAG/predictions.csv
Saved: /content/drive/MyDrive/Bangla_Agri_RAG/calibration_train_sample.csv
Saved: /content/drive/MyDrive/Bangla_Agri_RAG/router_threshold_search.csv


In [4]:
import re
import unicodedata
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score

OUT = Path("/content/drive/MyDrive/Bangla_Agri_RAG")

df = pd.read_csv(OUT / "predictions.csv").fillna("")

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)

def normalize(text):

    text = unicodedata.normalize("NFKC", str(text))

    text = text.translate(BN_TO_EN).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(r"\s+", " ", text).strip()


def tokens(text):
    return normalize(text).split()

def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum((Counter(p) & Counter(g)).values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return 2 * precision * recall / (precision + recall)


def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pg = Counter(tuple(p[i:i+n]) for i in range(len(p)-n+1))
    gg = Counter(tuple(g[i:i+n]) for i in range(len(g)-n+1))

    overlap = sum((pg & gg).values())

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pg.values())
    recall = overlap / sum(gg.values())

    return 2 * precision * recall / (precision + recall)

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(dp[j-1] + 1)
            else:
                new.append(max(dp[j], new[-1]))

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


df["Exact Match"] = [
    exact_match(p, g) for p, g in zip(df["prediction"], df["gold"])
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(normalize(p), normalize(g)) / 100
    for p, g in zip(df["prediction"], df["gold"])
]

df["Token F1"] = [
    token_f1(p, g) for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1) for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2) for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-L"] = [
    rouge_l(p, g) for p, g in zip(df["prediction"], df["gold"])
]

bleu = BLEU(tokenize="none", smooth_method="exp", effective_order=True)

pred_texts = [" ".join(tokens(x)) for x in df["prediction"]]
gold_texts = [" ".join(tokens(x)) for x in df["gold"]]

corpus_bleu = bleu.corpus_score(pred_texts, [gold_texts]).score / 100


print("Calculating multilingual BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=4,
    device="cpu",
    idf=False,
    rescale_with_baseline=False,
    verbose=True
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()


result = pd.DataFrame({
    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1",
        "Truncated Outputs"
    ],
    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean(),
        df["truncated"].astype(str).str.lower().eq("true").sum()
    ]
})

df.to_csv(OUT / "predictions.csv", index=False, encoding="utf-8-sig")

result.to_csv(OUT / "result.csv", index=False, encoding="utf-8-sig")

display(result)

print("\nSaved:")
print(OUT / "retrievals.csv")
print(OUT / "retrievals.json")
print(OUT / "predictions.csv")
print(OUT / "result.csv")


if "method" in df.columns:
    breakdown = df.groupby("method").agg(
        n=("method", "size"),
        exact_match=("Exact Match", "mean"),
        fuzzy_match=("Fuzzy Match", "mean"),
        rouge_l=("ROUGE-L", "mean"),
        token_f1=("Token F1", "mean"),
        bert_f1=("BERT F1", "mean"),
    ).reset_index()
    print("\nMetric breakdown by answering method:")
    display(breakdown)


Calculating multilingual BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

done in 15.36 seconds, 13.02 sentences/sec


,metric,score
0,Exact Match,0.545000
1,Fuzzy Match,0.872586
2,Corpus BLEU,0.639271
3,ROUGE-1,0.755459
4,ROUGE-2,0.709178
5,ROUGE-L,0.742539
6,Token F1,0.755459
7,BERT Precision,0.904453
8,BERT Recall,0.918609
9,BERT F1,0.910751



Saved:
/content/drive/MyDrive/Bangla_Agri_RAG/retrievals.csv
/content/drive/MyDrive/Bangla_Agri_RAG/retrievals.json
/content/drive/MyDrive/Bangla_Agri_RAG/predictions.csv
/content/drive/MyDrive/Bangla_Agri_RAG/result.csv

Metric breakdown by answering method:


,method,n,exact_match,fuzzy_match,rouge_l,token_f1,bert_f1
0,extractive,87,0.701149,0.863853,0.774296,0.785124,0.924859
1,generative,113,0.424779,0.879310,0.718090,0.732620,0.899890
